<a href="https://colab.research.google.com/github/wht-trc/ArcheAgeKillCountAnalyst/blob/main/ArcheAgeKillCountAnalyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **ArcheAge Kill-Count Analyst: AI-ассистент для аналитики PvP-праймов**

Данный ноутбук реализует агента на базе языковой модели DeepSeek, который анализирует статистику убийств из Google-таблиц.  
Ассистент отвечает на вопросы о результатах игроков, строит топы, считает средние значения и отслеживает историю ников.

**Как это работает:**
- Агент использует **один инструмент** (тулзу) для загрузки данных из таблицы.
- Можно задать вопрос вопрос на естественном языке (например, «Кто топ-1 за всё время?»), агент сам вызывает инструмент с нужными параметрами и возвращает ответ.
- В интерфейсе Gradio отображается история диалога и отчёт о выполнении каждого запроса (время, число вызовов LLM и инструментов).

**Практическая ценность:**
Инструмент экономит время глав гильдий и рядовых игроков, избавляя от ручного поиска и расчётов в гугл‑таблицах, и добавляет элемент фана.



---



## **Установка библиотек и получение доступа к DeepSeek API и Google Sheets API, подключение к таблице**

Сначала установим необходимые библиотеки.

In [1]:
# Установка необходимых библиотек и импорты
#!!pip uninstall -y langchain langchain-community langchain-core langchain-openai langchain-classic langgraph langgraph-prebuilt langgraph-sdk openai
!pip install -q langchain==0.3.13 langchain-openai gradio pandas gspread oauth2client

import os, time, json, re, pandas as pd
from datetime import datetime, timedelta

# Компоненты LangChain для создания агента
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain.tools import tool
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# Google Sheets API
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from google.colab import userdata

# Градио для интерактивного интерфейса
import gradio as gr

Получим API-ключ DeepSeek из секретов Coolab.

In [2]:
# Безопасное получение API-ключа DeepSeek из секретов Colab
DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')

Авторизуемся в Google Sheets API через сервисный аккаунт.

In [3]:
# Авторизация в Google Sheets API через сервисный аккаунт

# Получаем JSON-ключ сервисного аккаунта из секретов Colab
# (предварительно добавляем секрет GCP_SERVICE_ACCOUNT со всем содержимым файла credentials.json)
creds_json = userdata.get('GCP_SERVICE_ACCOUNT')
creds_dict = json.loads(creds_json)

# Настраиваем права доступа
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)

# Авторизуемся и открываем таблицу по ID
client = gspread.authorize(creds)
SPREADSHEET_ID = "1NHz6jDXi8XDpXL1OCpelKvgG3XB3G9LcHL9KdO958C8"
sheet = client.open_by_key(SPREADSHEET_ID)

# Выводим названия всех доступных листов (для проверки)
print("Доступные листы:", [ws.title for ws in sheet.worksheets()])

Доступные листы: ['июнь 2026', 'май 2026', 'апрель 2026', 'декабрь 2025', 'ноябрь 2025', 'октябрь 2025', 'сентябрь 2025', 'август 2025', 'июль 2025', 'июнь 2025', 'май 2025', 'апрель 2025', 'март 2025', 'февраль 2025', 'январь 2025', 'декабрь 2024', 'ноябрь 2024', 'октябрь 2024', 'август 2024', 'май 2024', 'апрель 2024', 'март 2024', 'февраль 2024', 'январь 2024', 'декабрь 2023', 'ноябрь 2023', 'октябрь 2023', 'август-сентябрь 2023']


## **Мониторинг: сбор статистики и выполнение запросов**

Класс мониторинга (AgentMonitor) собирает и отображает метрики выполнения каждого запроса:
- Общее время выполнения;
- Количество циклов рассуждения LLM;
- Количество обращений к источнику данных (Tool).

Отчет выводится в консоль и используется для отображения в Gradio-интерфейсе.

In [4]:
# Мониторинг: класс для сбора и отображения отчёта о выполнении запроса
class AgentMonitor:
    """
    Простой сборщик метрик: время выполнения, количество вызовов LLM и инструментов.
    """
    def __init__(self):
        self.start_time = 0.0
        self.end_time = 0.0
        self.llm_calls = 0 # Сколько раз обращались к LLM
        self.tool_calls = 0 # Сколько раз вызывалась функция-инструмент
        self.last_report = "" # Последний сформированный отчёт

    def log_start(self):
        # Засекаем время начала запроса
        self.start_time = time.time()

    def log_end(self):
        # Фиксируем время окончания запроса
        self.end_time = time.time()

    # Выводим отчет в консоль и сохраняем его в last_report
    def print_report(self):
        duration = self.end_time - self.start_time
        report = "\n" + "="*35 + "\n"
        report += "ОТЧЕТ О ВЫПОЛНЕНИИ ЗАПРОСА\n"
        report += "="*35 + "\n"
        report += f"Общее время выполнения: {duration:.2f} с\n"
        report += f"Циклов рассуждения (LLM): {self.llm_calls}\n"
        report += f"Обращений к источнику данных (Tool): {self.tool_calls}\n"
        report += "="*35 + "\n"
        print(report)
        self.last_report = report

# Глобальный объект мониторинга (будет заменяться для каждого нового запроса)
monitor = AgentMonitor()

## **Инструмент агента**

Инструмент для получения и анализа статистики праймов из Google-таблицы. Основная функция `get_kill_stats`.

**Параметры:**
  - `month`  - название листа (месяца), например "июнь 2026"
  - `player` - ник или часть ника (текущий, постоянный, на кириллице или латинице)
  - `date`   - конкретная дата в формате ДД.ММ.ГГГГ
  - `action` - тип запрашиваемой операции:
    - "raw"                – просто вывести все записи;
    - "player_avg"         – среднее значение метрики для игрока;
    - "player_total"       – суммарное значение метрики для игрока;
    - "top_avg"            – топ N игроков по среднему значению метрики;
    - "top_total"          – топ N игроков по сумме метрики;
    - "player_nicks_history" – история всех ников игрока.
  - `top_n`  - число записей для топов (по умолчанию 5)
  - `server` - фильтр по названию сервера (например "ЛУЦИЙ")
  - `period` - временной период: "last_year", "all_time" или "YYYY-MM-DD:YYYY-MM-DD"
  - `metric` - метрика для расчётов:
               "kills", "honor", "points", "efficiency" (очки/килл), "place" (место)

**Алгоритм работы:**
1. Выбирает нужные листы гугл-таблицы.
2. Загружает лист, обрабатывает построчно: определяет сервер, дату, заголовки и строки с данными.
3. Извлекает место, ник (текущий и постоянный), класс, киллы, хонор, очки.
4. Нормализует постоянный ник (нижний регистр) для объединения разных написаний.
5. Применяет фильтры (период, сервер, дата).
6. Ищет игрока (точный → префиксный → нечеткий поиск).
7. Выполняет запрошенное действие: вывод сырых данных, расчет среднего/суммарного для игрока, построение топов, историю ников.

In [5]:
# Инструмент агента: прямое чтение Google Таблицы и аналитика

@tool
def get_kill_stats(
    month: str = None,
    player: str = None,
    date: str = None,
    action: str = "raw",
    top_n: int = 5,
    server: str = None,
    period: str = None,
    metric: str = "kills"
) -> str:
    """
    Инструмент для получения и анализа статистики праймов из Google Таблицы.
    Параметры:
      month  - название листа (месяца), например "июнь 2026"
      player - ник или часть ника (текущий, постоянный, на кириллице или латинице)
      date   - конкретная дата в формате ДД.ММ.ГГГГ
      action - тип запрашиваемой операции
      top_n  - число записей для топов (по умолчанию 5)
      server - фильтр по названию сервера
      period - временной период: last_year, all_time или YYYY-MM-DD:YYYY-MM-DD
      metric - метрика: kills, honor, points, efficiency, place
    """
    global monitor
    monitor.tool_calls += 1

    try:
        # Выбор листов для обработки
        all_worksheets = sheet.worksheets()
        if month:
            month_lower = month.lower()
            worksheets = [ws for ws in all_worksheets if ws.title.lower() == month_lower]
            if not worksheets:
                available = [ws.title for ws in all_worksheets]
                return f"Месяц '{month}' не найден. Доступные: {available}"
        else:
            worksheets = all_worksheets

        records = []
        date_pattern = re.compile(r'^\d{2}\.\d{2}\.\d{4}$')
        HEADER_KEYWORDS = {"место", "ник", "класс", "киллы", "хонор", "очки"}

        # Обработка каждого листа
        for ws in worksheets:
            # Получаем все данные листа одной операцией
            data = ws.get_all_values()
            if not data:
                continue

            current_server = "Неизвестно"
            current_date = None

            for row in data:
                # Пропускаем полностью пустые строки
                if all(not str(cell).strip() for cell in row):
                    continue

                cells = [str(cell).strip() for cell in row]
                if len(cells) < 2:
                    continue

                second_cell = cells[1]

                # Дата прайма
                if date_pattern.match(second_cell):
                    current_date = second_cell
                    continue

                # Заголовок таблицы
                if second_cell.lower() in HEADER_KEYWORDS:
                    continue

                # Сервер
                if second_cell and not cells[2] if len(cells) > 2 else True:
                    if not any(kw in second_cell.lower() for kw in HEADER_KEYWORDS):
                        current_server = second_cell
                        continue

                # Строка с данными
                if len(cells) >= 7 and current_date:
                    if not cells[1].isdigit():
                        continue

                    try:
                        place = int(cells[1])
                        kills = int(re.sub(r'[^\d]', '', cells[4]) or 0)
                        honor = int(re.sub(r'[^\d]', '', cells[5]) or 0)
                        points = int(re.sub(r'[^\d]', '', cells[6]) or 0)

                        full_nick = cells[2]
                        nick_class = cells[3]

                        true_nick = full_nick
                        match = re.search(r'\((.*?)\)', full_nick)
                        if match:
                            true_nick = match.group(1)

                        normalized_true = true_nick.lower()

                        records.append({
                            "server": current_server,
                            "date": current_date,
                            "place": place,
                            "nickname": full_nick,
                            "true_nickname": true_nick,
                            "normalized_true": normalized_true,
                            "class": nick_class,
                            "kills": kills,
                            "honor": honor,
                            "points": points
                        })
                    except (ValueError, IndexError) as e:
                        print(f"Ошибка парсинга строки в листе {ws.title}: {e}")
                        print(f"  Ячейки: {cells}")
                        continue

        if not records:
            return "Данные не найдены."

        # DataFrame и фильтры
        df = pd.DataFrame(records)
        df["date_dt"] = pd.to_datetime(df["date"], format="%d.%m.%Y", errors="coerce")
        df = df.dropna(subset=["date_dt"])

        if period:
            now = datetime.now()
            if period == "last_year":
                start_date = now - timedelta(days=365)
                df = df[df["date_dt"] >= start_date]
            elif period == "all_time":
                pass
            elif ":" in period:
                parts = period.split(":")
                if len(parts) == 2:
                    try:
                        start = datetime.strptime(parts[0], "%Y-%m-%d")
                        end = datetime.strptime(parts[1], "%Y-%m-%d")
                        df = df[(df["date_dt"] >= start) & (df["date_dt"] <= end)]
                    except:
                        return "Неверный формат периода. Используйте YYYY-MM-DD:YYYY-MM-DD."

        if server:
            df = df[df["server"].str.contains(server, case=False, na=False)]
        if date:
            df = df[df["date"] == date]

        # Поиск игрока
        if player:
            player_lower = player.lower().strip()
            mask = (
                df["nickname"].str.lower().str.contains(player_lower, na=False) |
                df["true_nickname"].str.lower().str.contains(player_lower, na=False) |
                df["normalized_true"].str.lower().str.contains(player_lower, na=False)
            )
            filtered_df = df[mask]

            if filtered_df.empty:
                all_nicks = pd.concat([df["nickname"], df["true_nickname"], df["normalized_true"]]).unique()
                prefix_matches = [nick for nick in all_nicks if nick.lower().startswith(player_lower)]
                if prefix_matches:
                    suggestions = ", ".join(sorted(set(prefix_matches))[:5])
                    return (f"Точных совпадений для '{player}' не найдено. "
                            f"Возможно, вы имели в виду: {suggestions}?")
                else:
                    import difflib
                    close_matches = difflib.get_close_matches(
                        player_lower, [n.lower() for n in all_nicks], n=5, cutoff=0.6
                    )
                    if close_matches:
                        suggestions = ", ".join(close_matches)
                        return (f"Точных совпадений для '{player}' не найдено. "
                                f"Возможно, вы имели в виду: {suggestions}?")
                    else:
                        return (f"Игрок '{player}' не найден, и похожих ников не обнаружено. "
                                f"Проверьте написание или попробуйте другой ник.")
            df = filtered_df

        if df.empty:
            return "Данных по заданным параметрам нет."

        # Вывод (без технических столбцов
        display_columns = ["server", "date", "place", "nickname", "true_nickname",
                           "class", "kills", "honor", "points"]

        # Действия
        if action == "raw":
            return f"Найдено записей: {len(df)}\n" + df[display_columns].to_string(index=False, max_rows=25)

        elif action == "player_nicks_history":
            hist = df.groupby("normalized_true").apply(
                lambda x: x[["nickname", "date", "server"]].drop_duplicates().to_dict("records")
            ).reset_index(name="nicks")
            if hist.empty:
                return "Ники не найдены."
            result = ""
            for _, row in hist.iterrows():
                result += f"Постоянный ник: {row['normalized_true']}\n"
                for entry in row["nicks"]:
                    result += f"  - {entry['nickname']} (сервер: {entry['server']}, дата: {entry['date']})\n"
            return result

        elif action in ("player_avg", "player_total", "top_avg", "top_total"):
            if metric == "efficiency":
                df = df[df["kills"] > 0].copy()
                df["metric_value"] = df["points"] / df["kills"]
            elif metric == "place":
                df["metric_value"] = df["place"]
            else:
                col_map = {"kills": "kills", "honor": "honor", "points": "points"}
                if metric not in col_map:
                    return f"Неизвестная метрика: {metric}. Допустимые: kills, honor, points, efficiency, place."
                df["metric_value"] = df[col_map[metric]]

            if action == "player_avg":
                if not player:
                    return "Укажите игрока."
                daily = df.groupby("date")["metric_value"].mean()
                if daily.empty:
                    return "Нет данных."
                avg_val = daily.mean()
                total_val = daily.sum()
                days = len(daily)
                nicks = df["nickname"].unique()
                return (f"Игрок: {player} (найден как: {', '.join(nicks)})\n"
                        f"Метрика: {metric}\n"
                        f"Праймов с участием: {days}\n"
                        f"Сумма: {total_val:.2f}\n"
                        f"Среднее за прайм: {avg_val:.2f}")

            elif action == "player_total":
                total_val = df["metric_value"].sum()
                return f"Игрок {player}: суммарное значение {metric} = {total_val:.2f}"

            elif action == "top_avg":
                daily = df.groupby(["normalized_true", "date"])["metric_value"].mean().reset_index()
                avg = daily.groupby("normalized_true")["metric_value"].mean().reset_index()
                ascending = (metric == "place")
                avg = avg.sort_values("metric_value", ascending=ascending).head(top_n)
                result = f"Топ-{top_n} по среднему '{metric}':\n"
                for _, row in avg.iterrows():
                    result += f"{row['normalized_true']}: {row['metric_value']:.2f}\n"
                return result

            elif action == "top_total":
                total = df.groupby("normalized_true")["metric_value"].sum().reset_index()
                ascending = (metric == "place")
                total = total.sort_values("metric_value", ascending=ascending).head(top_n)
                result = f"Топ-{top_n} по сумме '{metric}':\n"
                for _, row in total.iterrows():
                    result += f"{row['normalized_true']}: {row['metric_value']:.2f}\n"
                return result

        return f"Неизвестное действие: {action}"

    except Exception as e:
        return f"Ошибка при обработке данных: {str(e)}"

## **Настройка языковой модели DeepSeek**

In [6]:
# Настройка языковой модели (DeepSeek)

llm = ChatOpenAI(
    model="deepseek-chat",
    openai_api_key=DEEPSEEK_API_KEY,
    openai_api_base="https://api.deepseek.com/v1",
    temperature=0.1,      # низкая температура для точности
    max_tokens=600
)

## **Системный промпт**

Описывает роль, контекст, инструкции, примеры и формат ответа.  
Промпт заставляет агента **всегда** вызывать инструмент для вопросов о статистике, использовать средние значения для топов (если не указано «суммарные»), передавать даты в правильном формате и корректно обрабатывать ситуацию «игрок не найден».

In [7]:
# Системный промпт агента: контекст, инструкции, примеры, формат ответа
system_message = """
Ты — летописец PvP-праймов ArcheAge, ассистент гильдии.

### Контекст
- Статистика хранится в Google Таблицах, разбитых по месяцам (например, "июнь 2026").
- Никнейм может быть в формате "ТекущийНик (НастоящийНик)". НастоящийНик — постоянный идентификатор игрока.
- Доступны метрики: kills (убийства), honor (честь), points (очки), efficiency (очки/килл), place (место).
- Периоды: last_year, all_time, или точный интервал YYYY-MM-DD:YYYY-MM-DD.

### Инструкции
1. Для **любого вопроса о статистике** обязательно вызывай инструмент `get_kill_stats`.
2. Если запрашивают **лучшего / топ‑1 / самого результативного** БЕЗ слов «суммарно», «всего» — используй **среднее значение** (action="top_avg", metric="kills").
3. Только при явном указании «суммарные» / «всего убийств» применяй action="top_total".
4. Даты передавай строго в формате "ДД.ММ.ГГГГ".
5. Если игрок не найден — получишь от инструмента подсказки с похожими никами; либо переспроси пользователя, либо уточни по первой же подсказке.

### Примеры
| Вопрос пользователя | Ожидаемый вызов инструмента |
|---------------------|------------------------------|
| «Кто топ-1 за всё время?» | get_kill_stats(action="top_avg", metric="kills", period="all_time", top_n=1) |
| «У кого больше всего суммарных очков в июне?» | get_kill_stats(action="top_total", metric="points", month="июнь 2026", top_n=1) |
| «Какие ники были у Лакшери?» | get_kill_stats(action="player_nicks_history", player="Лакшери", period="all_time") |
| «Среднее число убийств у Кайфова за последний год» | get_kill_stats(action="player_avg", metric="kills", player="Кайфова", period="last_year") |
| «Дай записи за 07.06.2026» | get_kill_stats(action="raw", date="07.06.2026") |

### Формат ответа
- Отвечай на русском, кратко (1-3 предложения), но содержательно.
- Если данных нет — так и скажи, не додумывай.
- В конце можно добавить небольшой аналитический комментарий, если он уместен.
"""

# Собираем шаблон промпта
prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

## **Создание агента**

In [8]:
# Сборка агента: инструмент + LLM + промпт

tools = [get_kill_stats]

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True, # Вывод логов в Colab
    handle_parsing_errors=True,
    max_iterations=5 # Максимальное число вызовов инструмента
)

## **Функция обработки запроса с мониторингом**

`process_query` принимает вопрос пользователя, передаёт его агенту и возвращает кортеж (ответ, отчет о выполнении).  
Перед вызовом агента создаётся новый объект `AgentMonitor`, который используется и внутри инструмента.  
После выполнения запроса отчет выводится в консоль и возвращается для отображения в Gradio.

In [9]:
# Функция обработки запроса с мониторингом (исправленная телеметрия)
# Обработка вопроса от пользователя и сбор отчета о выполнении

def process_query(user_input: str) -> tuple:
    """
    Принимает текст вопроса, возвращает (ответ_агента, отчёт_о_выполнении).
    """

    # Создаём локальный монитор для этого запроса
    local_monitor = AgentMonitor()
    local_monitor.log_start()

    # Передаем его в инструмент через глобальную переменную
    global monitor
    monitor = local_monitor

    try:
        result = agent_executor.invoke({"input": user_input})
        answer = result["output"]

        # Оцениваем количество вызовов LLM по числу промежуточных шагов
        if "intermediate_steps" in result:
            local_monitor.llm_calls = len(result["intermediate_steps"])
        else:
            # Оцениваем минимально: 1 вызов на каждый ответ
            local_monitor.llm_calls = 1

    except Exception as e:
        answer = f"Произошла ошибка: {str(e)}"
        local_monitor.llm_calls = 1

    local_monitor.log_end()
    local_monitor.print_report() # печатаем отчет в консоль и сохраняем в last_report

    return answer, local_monitor.last_report

## **Gradio-интерфейс**

Создаём интерфейс с чатом и блоком отчета о выполнении.  

In [10]:
# Интерфейс Gradio с кастомным дизайном (диалог + отчет)

custom_css = """
    /* Общий фон всего приложения */
    .gradio-container {
        background-color: #1a1a2e;  /* тёмно-синий */
    }

    /* Заголовок (Markdown) */
    h1 {
        color: #ffffff;
        font-family: 'Segoe UI', sans-serif;
    }

    .custom-desc {
      color: #ffffff !important;
      font-size: 16px;
      margin-bottom: 10px;
    }

    .telemetry-header {
        color: #ffffff !important;
        margin-bottom: 5px;
    }

    /* Окно диалога (чат) */
    .chatbot {
        height: 500px;              /* высота чата */
        background-color: #16213e;  /* фон чата */
        border: 1px solid #0f3460;
    }

    /* Сообщения внутри чата */
    .message {
        font-size: 16px;
    }

    /* Сообщение пользователя */
    .message.user {
        background-color: #cdd0d4;
        color: #000000;
        border-radius: 10px;
    }

    /* Сообщение ассистента (бота) */
    .message.bot {
        background-color: #cdd0d4;
        color: #000000;
        border-radius: 10px;
    }

    /* Поле ввода вопроса */
    input[type="text"], textarea {
        font-size: 16px;
        background-color: #cdd0d4;
        color: #000000;
        border: none;
    }

    /* Кнопка "Отправить" */
    button.primary {
        background-color: #21a038;
        color: #ffffff;
        border: 1px solid #125a20;
        font-weight: bold;
    }

    /* Кнопка "Очистить историю" */
    button.secondary {
        background-color: #21a038;
        color: #ffffff;
        border: 1px solid #125a20;
    }

    /* Блок с телеметрией */
    .telemetry-box textarea {
        font-family: 'Courier New', monospace;
        font-size: 14px;
        background-color: #cdd0d4;
        color: #000000;
        border: 1px solid #000000;
    }

    /* Полоса прокрутки (для чата и телеметрии) */
    ::-webkit-scrollbar {
        width: 8px;
    }
    ::-webkit-scrollbar-track {
        background: #1a1a2e;
    }
    ::-webkit-scrollbar-thumb {
        background: #60635f;
        border-radius: 4px;
    }
"""

with gr.Blocks(css=custom_css, title="ArcheAge Kill-Count Analyst") as demo:
    gr.Markdown("# 🛡️ ArcheAge Kill‑Count Analyst")
    # Описание через HTML для точного применения стилей
    gr.HTML('<div class="custom-desc">Задавайте свои вопросы о статистике праймов.</div>')

    with gr.Row():
        with gr.Column(scale=3):
            # Чат с историей диалога
            chatbot = gr.Chatbot(label="Диалог", height=500, type="messages")
            # Строка ввода и кнопка
            with gr.Row(elem_classes="input-row"):
                user_input = gr.Textbox(
                    placeholder="Введите вопрос...",
                    label="Ваш вопрос",
                    scale=4
                )
                send_btn = gr.Button("Отправить", variant="primary", scale=1)

        with gr.Column(scale=1):
            # Блок с отчетом о выполнении запроса
            gr.HTML('<h3 class="telemetry-header">📊 Отчет о выполнении</h3>')
            telemetry_output = gr.Textbox(
                label="Отчет",
                lines=10,
                max_lines=15,
                interactive=False,
                elem_classes="report-box"
            )
            clear_btn = gr.Button("Очистить историю", variant="secondary")

    # Функция отправки сообщения
    def respond(history, message):
        # Добавляем сообщение пользователя в историю
        history.append({"role": "user", "content": message})
        # Вызываем агента
        answer, telemetry = process_query(message)
        # Добавляем ответ ассистента в историю
        history.append({"role": "assistant", "content": answer})
        # Возвращаем обновлённый диалог, очищенную строку ввода и отчет
        return history, "", telemetry

    # Очистка истории и отчета
    def clear_history():
        return [], ""

    # Привязка событий
    send_btn.click(
        respond,
        inputs=[chatbot, user_input],
        outputs=[chatbot, user_input, telemetry_output]
    )
    user_input.submit(
        respond,
        inputs=[chatbot, user_input],
        outputs=[chatbot, user_input, telemetry_output]
    )
    clear_btn.click(
        clear_history,
        inputs=[],
        outputs=[chatbot, telemetry_output]
    )

demo.launch(debug=True)

/tmp/ipykernel_75786/2391084687.py:97: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css, title="ArcheAge Kill-Count Analyst") as demo:
/tmp/ipykernel_75786/2391084687.py:105: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Диалог", height=500, type="messages")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://01f7c91eb89af92395.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)




> Entering new AgentExecutor chain...

Invoking: `get_kill_stats` with `{'action': 'top_avg', 'metric': 'kills', 'period': 'all_time', 'top_n': 1}`
responded: Пользователь спрашивает про топ-1 за всё время без уточнения «суммарно», поэтому используем среднее значение по убийствам.

Топ-1 по среднему 'kills':
божехранироссию: 191.60
🏆 **Топ-1 за всё время по среднему числу убийств — божехранироссию** со средним показателем **191.60 убийств** за прайм. Достойный результат, настоящий снайпер! 🔥

> Finished chain.

ОТЧЕТ О ВЫПОЛНЕНИИ ЗАПРОСА
Общее время выполнения: 10.40 с
Циклов рассуждения (LLM): 1
Обращений к источнику данных (Tool): 1

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://01f7c91eb89af92395.gradio.live
